# Segmentasi Nasabah Bank — dari Transaksi ke Perilaku Nasabah

**Pertanyaan bisnis:** *Kelompok nasabah seperti apa yang ada di data kita, dan tindakan apa yang masuk akal untuk tiap kelompok?*

**Alur kerja (dan alasan tiap langkah):**

| # | Tahap | Pertanyaan yang dijawab |
|---|-------|--------------------------|
| 1 | Audit data | Apa yang sebenarnya ada di data? Apa unit barisnya? |
| 2 | Pembersihan | Apa yang rusak, dan apa yang *tidak boleh* dibuang? |
| 3 | Feature engineering | Bagaimana mengubah transaksi menjadi **perilaku nasabah**? |
| 4 | Pemilihan k & validasi | Apakah cluster ini nyata, atau hanya hasil algoritma memaksa memotong data? |
| 5 | Profiling segmen | Apa arti tiap segmen secara bisnis? |
| 6 | Klasifikasi (surrogate) | Bisakah nasabah baru dialokasikan ke segmen dengan cepat dan dapat dijelaskan? |
| 7 | Ekspor artefak | Apa yang dibutuhkan dashboard / deployment? |

> **Catatan perbaikan utama dari versi awal:** (1) unit analisis diganti dari *transaksi* ke *nasabah*, (2) semua fitur diskalakan sebelum KMeans (versi awal: `LocationCoords` berskala ribuan menguasai seluruh jarak), (3) outlier `LoginAttempts` tidak dibuang karena itu sinyal risiko, (4) cluster divalidasi dengan metrik tambahan dan pembanding data acak, (5) yang disimpan adalah **model** (pipeline), bukan data.

## 0. Konfigurasi

In [ ]:
import json
import platform
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, adjusted_rand_score,
                             calinski_harabasz_score, classification_report,
                             davies_bouldin_score, f1_score, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_validate,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

# Sumber data: taruh file CSV di folder yang sama, atau biarkan membaca dari link Drive yang sama dengan notebook awal.
DATA_PATH = "bank_transactions_data.csv"
DATA_URL = "https://drive.google.com/uc?id=1gnLO9qvEPqv1uBt1928AcsCmdvzqjC5m"

K_OVERRIDE = None  # isi angka (mis. 4) jika ingin memaksa jumlah segmen setelah melihat hasil validasi

## 1. Audit data

**Kenapa dulu audit, bukan langsung membersihkan?** Keputusan pembersihan yang benar hanya bisa diambil kalau kita tahu
(a) apa arti satu baris, (b) pola missing value-nya, dan (c) kolom mana yang punya variasi.

Pertanyaan kunci: **satu baris = satu transaksi, tetapi tujuan kita segmentasi *nasabah*.** Apakah satu nasabah punya banyak baris?

In [ ]:
raw = pd.read_csv(DATA_PATH) if Path(DATA_PATH).exists() else pd.read_csv(DATA_URL)

print("Shape:", raw.shape)
display(raw.head())

audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_missing": raw.isna().sum(),
    "pct_missing": (raw.isna().mean() * 100).round(2),
    "n_unique": raw.nunique(),
})
display(audit)

n_acc = raw["AccountID"].nunique()
print("Baris duplikat penuh   :", raw.duplicated().sum())
print("Jumlah akun unik       :", n_acc, "| rata-rata transaksi per akun:", round(len(raw) / n_acc, 2))
print("Sebaran LoginAttempts  :", raw["LoginAttempts"].value_counts(dropna=False).sort_index().to_dict())
print("Baris dengan >=1 NaN   :", raw.isna().any(axis=1).sum(), f"({raw.isna().any(axis=1).mean():.0%} dari data)")

**Cara membaca output:**
- Jika `rata-rata transaksi per akun` > 1, maka segmentasi yang benar dilakukan pada level **akun** (satu akun = satu nasabah), bukan per transaksi. Satu nasabah tidak boleh muncul di tiga segmen berbeda hanya karena tiga transaksinya berbeda.
- `Baris dengan >=1 NaN` menunjukkan kenapa `dropna()` pada seluruh kolom berbahaya: missing value yang tersebar di banyak kolom (masing-masing hanya beberapa persen) bisa menghapus sebagian besar baris.
- `LoginAttempts` > 1 biasanya jarang. Itu **bukan sampah**, itu kandidat sinyal risiko.

## 2. Pembersihan data (level transaksi)

Prinsip yang dipakai:
1. **Buang hanya yang tidak bisa diselamatkan:** duplikat persis, dan baris tanpa kunci (`AccountID`, `TransactionAmount`, `TransactionDate`).
2. **Imputasi** sisanya: numerik dengan median akun (fallback median global), kategorikal dengan modus akun (fallback modus global).
3. **Tidak membuang outlier.** Outlier di data transaksi sering justru kejadian yang paling menarik (transaksi besar, banyak percobaan login). Skewness ditangani dengan transformasi log, dan kejadian ekstrem dijadikan fitur.

In [ ]:
NUM_COLS = ["TransactionAmount", "CustomerAge", "TransactionDuration", "LoginAttempts", "AccountBalance"]
CAT_COLS = ["TransactionType", "Location", "Channel", "CustomerOccupation"]
KEY_COLS = ["AccountID", "TransactionAmount", "TransactionDate"]

tx = raw.drop_duplicates().copy()
n_start = len(tx)

tx["TransactionDate"] = pd.to_datetime(tx["TransactionDate"], errors="coerce")
tx["PreviousTransactionDate"] = pd.to_datetime(tx["PreviousTransactionDate"], errors="coerce")
tx = tx.dropna(subset=KEY_COLS).copy()
n_after_keys = len(tx)

for c in NUM_COLS:
    acc_median = tx.groupby("AccountID")[c].transform("median")
    tx[c] = tx[c].fillna(acc_median).fillna(tx[c].median())

for c in CAT_COLS:
    acc_mode = tx.groupby("AccountID")[c].transform(lambda s: s.mode().iat[0] if s.notna().any() else np.nan)
    tx[c] = tx[c].fillna(acc_mode).fillna(tx[c].mode().iat[0])

print(f"Setelah drop duplikat           : {n_start} baris")
print(f"Setelah drop baris tanpa kunci  : {n_after_keys} baris (hilang {n_start - n_after_keys})")
print("Sisa NaN di kolom inti          :", int(tx[NUM_COLS + CAT_COLS].isna().sum().sum()))

## 3. Feature engineering: dari transaksi ke perilaku nasabah

**Cara berpikirnya:** segmentasi harus menjawab *"nasabah ini berperilaku seperti apa?"*. Maka fitur disusun dalam kerangka yang akrab di industri (mirip RFM):

| Dimensi | Fitur |
|---------|-------|
| Frekuensi | `n_txn`, `avg_gap_days` |
| Nilai uang | `avg_amount`, `avg_balance`, `avg_amount_to_balance` (seberapa besar transaksi dibanding saldo) |
| Kanal & jenis | `online_share`, `atm_share`, `debit_share` |
| Interaksi / risiko | `avg_duration`, `multi_login_rate`, `n_locations` |

Usia dan pekerjaan **sengaja tidak dimasukkan ke clustering**. Prinsipnya: *cluster berdasarkan perilaku, lalu profilkan dengan demografi.*
Dengan begitu kita bisa menguji hal yang bermakna: apakah demografi memprediksi perilaku?

Pengganti `LocationCoords` (jarak ke New York): jarak ke satu kota acuan yang sembarang tidak punya makna bisnis. `n_locations` (berapa lokasi berbeda yang dipakai nasabah) lebih relevan sebagai sinyal mobilitas/risiko.

In [ ]:
tx = tx.sort_values(["AccountID", "TransactionDate"]).reset_index(drop=True)

tx["amount_to_balance"] = tx["TransactionAmount"] / tx["AccountBalance"].clip(lower=1)
tx["gap_days"] = (tx["TransactionDate"] - tx["PreviousTransactionDate"]).dt.days.clip(lower=0)
tx["gap_days"] = tx["gap_days"].fillna(tx["gap_days"].median())
tx["multi_login"] = (tx["LoginAttempts"] > 1).astype(int)
tx["is_online"] = (tx["Channel"] == "Online").astype(int)
tx["is_atm"] = (tx["Channel"] == "ATM").astype(int)
tx["is_branch"] = (tx["Channel"] == "Branch").astype(int)
tx["is_debit"] = (tx["TransactionType"] == "Debit").astype(int)

g = tx.groupby("AccountID")
cust = pd.DataFrame({
    "n_txn": g.size(),
    "total_amount": g["TransactionAmount"].sum(),
    "avg_amount": g["TransactionAmount"].mean(),
    "avg_balance": g["AccountBalance"].mean(),
    "avg_amount_to_balance": g["amount_to_balance"].mean(),
    "avg_duration": g["TransactionDuration"].mean(),
    "multi_login_rate": g["multi_login"].mean(),
    "online_share": g["is_online"].mean(),
    "atm_share": g["is_atm"].mean(),
    "branch_share": g["is_branch"].mean(),
    "debit_share": g["is_debit"].mean(),
    "n_locations": g["Location"].nunique(),
    "avg_gap_days": g["gap_days"].mean(),
    "age": g["CustomerAge"].median(),
    "occupation": g["CustomerOccupation"].agg(lambda s: s.mode().iat[0]),
})

CLUSTER_FEATURES = ["n_txn", "avg_amount", "avg_balance", "avg_amount_to_balance", "avg_duration",
                    "multi_login_rate", "online_share", "atm_share", "debit_share",
                    "n_locations", "avg_gap_days"]
LOG_FEATURES = ["n_txn", "avg_amount", "avg_balance", "avg_amount_to_balance", "avg_gap_days"]  # skewed -> log1p

# Fitur tanpa variasi tidak bermakna untuk clustering (di versi awal LoginAttempts bernilai 1 untuk semua baris).
zero_var = [c for c in CLUSTER_FEATURES if cust[c].nunique() <= 1]
if zero_var:
    print("Fitur dibuang karena konstan:", zero_var)
CLUSTER_FEATURES = [c for c in CLUSTER_FEATURES if c not in zero_var]
LOG_FEATURES = [c for c in LOG_FEATURES if c in CLUSTER_FEATURES]

print("Jumlah nasabah:", len(cust))
display(cust[CLUSTER_FEATURES].describe().T.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cust[CLUSTER_FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0], annot_kws={"size": 7})
axes[0].set_title("Korelasi antar fitur clustering")
cust["n_txn"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Jumlah transaksi per nasabah"); axes[1].set_xlabel("n_txn"); axes[1].set_ylabel("jumlah nasabah")
plt.tight_layout(); plt.show()

corr = cust[CLUSTER_FEATURES].corr().abs()
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(corr.index) for b in corr.columns[i + 1:] if corr.loc[a, b] > 0.8]
print("Pasangan fitur berkorelasi > 0.8 (pertimbangkan membuang salah satu):", pairs or "tidak ada")

## 4. Memilih jumlah segmen (k) dan **memvalidasi** bahwa cluster itu nyata

Satu metrik (elbow) tidak cukup. Elbow sering ambigu, dan KMeans akan *selalu* menghasilkan k kelompok, bahkan dari data acak.
Maka kita gunakan empat sudut pandang + satu pembanding:

- **Silhouette** (lebih tinggi = lebih baik), **Davies–Bouldin** (lebih rendah = lebih baik), **Calinski–Harabasz** (lebih tinggi = lebih baik)
- **Stabilitas bootstrap (ARI)**: jika data diambil ulang, apakah segmennya kurang lebih sama? Mendekati 1 = stabil.
- **Pembanding data acak**: silhouette pada data yang kolomnya diacak independen (struktur multivariat dihancurkan). Jika silhouette asli tidak jauh di atas ini, *struktur cluster-nya lemah*.

Preprocessing dibungkus dalam satu `Pipeline`: log-transform untuk fitur miring, lalu `StandardScaler` untuk **semua** fitur.

In [ ]:
def make_prep(features, log_features):
    log_cols = [c for c in features if c in log_features]
    other_cols = [c for c in features if c not in log_features]
    transformers = []
    if log_cols:
        transformers.append(("log", Pipeline([("log", FunctionTransformer(np.log1p)), ("sc", StandardScaler())]), log_cols))
    if other_cols:
        transformers.append(("std", StandardScaler(), other_cols))
    return ColumnTransformer(transformers)


prep = make_prep(CLUSTER_FEATURES, LOG_FEATURES)
X_scaled = prep.fit_transform(cust[CLUSTER_FEATURES])
print("Matriks fitur terskala:", X_scaled.shape, "| std per kolom ~", np.round(X_scaled.std(axis=0).mean(), 2))


def bootstrap_stability(X, k, n_boot=30, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    ref = KMeans(k, n_init=20, random_state=seed).fit(X)
    aris = []
    for b in range(n_boot):
        idx = rng.integers(0, len(X), len(X))
        km = KMeans(k, n_init=10, random_state=b).fit(X[idx])
        aris.append(adjusted_rand_score(ref.labels_, km.predict(X)))
    return float(np.mean(aris)), float(np.std(aris))


rows = []
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=RANDOM_STATE).fit(X_scaled)
    stab, stab_sd = bootstrap_stability(X_scaled, k)
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(X_scaled, km.labels_),
                 "davies_bouldin": davies_bouldin_score(X_scaled, km.labels_),
                 "calinski_harabasz": calinski_harabasz_score(X_scaled, km.labels_),
                 "stability_ARI": stab, "min_segment_pct": np.bincount(km.labels_).min() / len(X_scaled) * 100})
sweep = pd.DataFrame(rows).set_index("k")
display(sweep.round(3))

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col, title in zip(axes, ["inertia", "silhouette", "davies_bouldin", "stability_ARI"],
                          ["Elbow (inertia)", "Silhouette (tinggi = baik)", "Davies-Bouldin (rendah = baik)", "Stabilitas bootstrap (ARI)"]):
    sweep[col].plot(marker="o", ax=ax); ax.set_title(title); ax.set_xlabel("k")
plt.tight_layout(); plt.show()

In [ ]:
# Pilih k: silhouette tertinggi pada rentang yang masuk akal secara bisnis (3-6 segmen). Bisa di-override di sel konfigurasi.
candidate = sweep.loc[3:6]
K = int(K_OVERRIDE) if K_OVERRIDE else int(candidate["silhouette"].idxmax())
print("k terpilih:", K)


def null_silhouette(X, k, n=20, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        Xp = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
        lab = KMeans(k, n_init=5, random_state=0).fit_predict(Xp)
        vals.append(silhouette_score(Xp, lab))
    return np.array(vals)


null_vals = null_silhouette(X_scaled, K)
real_sil = sweep.loc[K, "silhouette"]
print(f"Silhouette data asli      : {real_sil:.3f}")
print(f"Silhouette data teracak   : {null_vals.mean():.3f} +/- {null_vals.std():.3f}")
print(f"Selisih (asli - acak)     : {real_sil - null_vals.mean():.3f}")
print(f"Stabilitas bootstrap (ARI): {sweep.loc[K, 'stability_ARI']:.3f}")

plt.figure(figsize=(7, 3.5))
plt.hist(null_vals, bins=10, color="lightgray", label="data teracak")
plt.axvline(real_sil, color="crimson", label="data asli"); plt.legend(); plt.title(f"Silhouette asli vs acak (k={K})")
plt.tight_layout(); plt.show()

**Cara menafsirkan (tulis kesimpulan Anda sendiri di sel ini setelah melihat angkanya):**

| Hasil | Arti | Cara menulisnya di laporan |
|-------|------|----------------------------|
| Silhouette > 0.5, jauh di atas data acak, ARI > 0.8 | Cluster alami & kuat | "Terdapat segmen yang terpisah jelas" |
| Silhouette 0.25–0.5, di atas data acak, ARI > 0.6 | Struktur ada tapi tumpang tindih | "Segmen berguna sebagai pengelompokan praktis, batasnya kabur" |
| Silhouette < 0.25 atau mendekati data acak, ARI rendah | **Tidak ada struktur cluster alami** | "Data tidak membentuk kelompok alami; segmen bersifat deskriptif (partisi), bukan temuan" |

Menulis kesimpulan ketiga dengan jujur **bukan kegagalan**. Itu justru menunjukkan Anda tidak menjual hasil yang tidak ada. Data transaksi sintetis sering berada di baris ketiga.

## 5. Membandingkan algoritma

KMeans mengasumsikan cluster berbentuk bola dengan ukuran serupa. Kita uji apakah algoritma lain memberi gambaran yang sama. Jika ARI antar algoritma tinggi, kesimpulan tidak bergantung pada pilihan algoritma.

In [ ]:
km_labels = KMeans(K, n_init=20, random_state=RANDOM_STATE).fit_predict(X_scaled)
gmm_labels = GaussianMixture(K, covariance_type="diag", n_init=5, random_state=RANDOM_STATE).fit_predict(X_scaled)
agg_labels = AgglomerativeClustering(K, linkage="ward").fit_predict(X_scaled)

compare = pd.DataFrame({
    "silhouette": {n: silhouette_score(X_scaled, l) for n, l in [("KMeans", km_labels), ("GMM (diag)", gmm_labels), ("Agglomerative (ward)", agg_labels)]},
    "davies_bouldin": {n: davies_bouldin_score(X_scaled, l) for n, l in [("KMeans", km_labels), ("GMM (diag)", gmm_labels), ("Agglomerative (ward)", agg_labels)]},
    "ARI_vs_KMeans": {"KMeans": 1.0, "GMM (diag)": adjusted_rand_score(km_labels, gmm_labels), "Agglomerative (ward)": adjusted_rand_score(km_labels, agg_labels)},
})
display(compare.round(3))

## 6. Model final & profiling segmen

Model yang disimpan adalah **pipeline** (preprocessing + KMeans), sehingga nasabah baru bisa langsung dialokasikan ke segmen dengan `pipeline.predict(df_baru)`.
(Di versi awal, `joblib.dump(X_cluster, ...)` menyimpan *data*, bukan model, sehingga tidak bisa dipakai memprediksi apa pun.)

In [ ]:
seg_pipe = Pipeline([
    ("prep", make_prep(CLUSTER_FEATURES, LOG_FEATURES)),
    ("km", KMeans(n_clusters=K, n_init=50, random_state=RANDOM_STATE)),
]).fit(cust[CLUSTER_FEATURES])

cust["segment"] = seg_pipe.predict(cust[CLUSTER_FEATURES])
sizes = cust["segment"].value_counts().sort_index()
print("Ukuran segmen:\n", pd.DataFrame({"n": sizes, "pct": (sizes / len(cust) * 100).round(1)}))
if (sizes / len(cust)).min() < 0.05:
    print("PERINGATAN: ada segmen < 5% populasi, pertimbangkan k lebih kecil.")

X_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_scaled)
coords = X_2d.transform(X_scaled)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=cust["segment"], palette="tab10", s=40, alpha=0.8)
plt.title(f"Segmen pada ruang PCA 2D (varians dijelaskan: {X_2d.explained_variance_ratio_.sum():.0%})")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout(); plt.show()

In [ ]:
PROFILE_COLS = CLUSTER_FEATURES + ["total_amount", "age"]

profile = cust.groupby("segment")[PROFILE_COLS].mean()
z_profile = (profile - cust[PROFILE_COLS].mean()) / cust[PROFILE_COLS].std()   # seberapa jauh dari rata-rata keseluruhan

plt.figure(figsize=(12, 0.6 * K + 3))
sns.heatmap(z_profile.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0, cbar_kws={"label": "z-score vs rata-rata populasi"})
plt.title("Profil segmen: merah = di atas rata-rata, biru = di bawah"); plt.tight_layout(); plt.show()

print("Rata-rata dalam satuan ASLI (bukan terskala):")
display(profile.round(2))


def auto_label(row, top=2):
    top_feats = row.abs().sort_values(ascending=False).head(top).index
    return ", ".join(f"{f} {'tinggi' if row[f] > 0 else 'rendah'}" for f in top_feats)


auto_names = {s: auto_label(z_profile.loc[s]) for s in z_profile.index}
for s, name in auto_names.items():
    print(f"Segmen {s}: {name}")

print("\nKomposisi pekerjaan per segmen (%):")
display((pd.crosstab(cust["segment"], cust["occupation"], normalize="index") * 100).round(1))

**Tugas Anda setelah menjalankan sel di atas:** beri nama bisnis tiap segmen pada `SEGMENT_NAMES` di bawah, lalu tulis rekomendasi tindakan.
Contoh format (isi dengan temuan Anda, jangan disalin):

- *Segmen X, "nama":* ciri utama (angka) → risiko/peluang → tindakan yang diusulkan → metrik keberhasilan.

In [ ]:
# Ganti dengan nama bisnis setelah Anda membaca heatmap & tabel profil.
SEGMENT_NAMES = {s: f"Segmen {s}" for s in sorted(cust["segment"].unique())}
cust["segment_name"] = cust["segment"].map(SEGMENT_NAMES)

## 7. Klasifikasi: apa gunanya, dan apa yang *tidak* dibuktikannya

Di versi awal, label segmen diprediksi **tanpa** `LocationCoords`, padahal label itu *dibentuk* oleh `LocationCoords`.
Akibatnya akurasi ≈ tebakan acak (0,38 vs baseline 0,40). Itu bukan "model kurang bagus", melainkan **target yang tidak bisa dipelajari dari fitur yang diberikan**.
Pengujian pada data awal Anda: dengan `LocationCoords` akurasi 1,00; tanpa itu 0,38.

Ada dua pertanyaan berbeda yang sah, dan keduanya dijawab di bawah:

- **A. Surrogate model** (fitur sama dengan clustering): bisakah segmen ditiru oleh model yang cepat & dapat dijelaskan? Akurasi tinggi di sini **diharapkan** dan hanya membuktikan bahwa model bisa dipakai untuk produksi/penjelasan. **Ini bukan bukti bahwa segmennya valid** (validitas dibuktikan di bagian 4).
- **B. Prediktabilitas dari demografi** (usia + pekerjaan saja): bisakah kita menebak segmen perilaku dari informasi yang sudah ada saat nasabah mendaftar? Ini pertanyaan bisnis yang nyata dan hasilnya bisa jujur rendah.

Evaluasi memakai `macro-F1` (adil untuk segmen berukuran berbeda), pembanding `DummyClassifier`, dan satu kali evaluasi akhir pada test set.

In [ ]:
X = cust[CLUSTER_FEATURES]
y = cust["segment"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def clf_pipe(model):
    return Pipeline([("prep", make_prep(CLUSTER_FEATURES, LOG_FEATURES)), ("clf", model)])


models = {
    "Baseline (mayoritas)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "SVM (RBF)": SVC(random_state=RANDOM_STATE),
}

rows = []
for name, m in models.items():
    r = cross_validate(clf_pipe(m), X_tr, y_tr, cv=cv, scoring={"acc": "accuracy", "f1": "f1_macro"})
    rows.append({"model": name, "cv_accuracy": r["test_acc"].mean(), "cv_macro_f1": r["test_f1"].mean(), "cv_f1_std": r["test_f1"].std()})
cv_table = pd.DataFrame(rows).set_index("model").sort_values("cv_macro_f1", ascending=False)
display(cv_table.round(3))

In [ ]:
# Tuning: pipeline + ruang pencarian kecil & terarah. (Tuning SVM yang berjam-jam biasanya terjadi karena fitur tak diskalakan
# dan rentang C/gamma terlalu lebar; di sini skala sudah ditangani pipeline.)
param_dists = {
    "Logistic Regression": {"clf__C": list(np.logspace(-2, 2, 20))},
    "Decision Tree": {"clf__max_depth": [2, 3, 4, 5, 6, 8], "clf__min_samples_leaf": [1, 3, 5, 10, 20]},
    "Random Forest": {"clf__n_estimators": [100, 200, 400], "clf__max_depth": [None, 4, 6, 8, 12], "clf__min_samples_leaf": [1, 2, 5]},
    "KNN": {"clf__n_neighbors": list(range(3, 26, 2)), "clf__weights": ["uniform", "distance"]},
    "SVM (RBF)": {"clf__C": list(np.logspace(-1, 2, 10)), "clf__gamma": ["scale", 0.01, 0.05, 0.1]},
}

best_name = cv_table.drop(index="Baseline (mayoritas)").index[0]
print("Model terbaik di CV (di luar baseline):", best_name)

search = RandomizedSearchCV(clf_pipe(models[best_name]), param_dists[best_name], n_iter=15, cv=cv,
                            scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_tr, y_tr)
print("Parameter terbaik:", search.best_params_)

untuned = clf_pipe(models[best_name]).fit(X_tr, y_tr)
dummy = clf_pipe(models["Baseline (mayoritas)"]).fit(X_tr, y_tr)
tuned = search.best_estimator_

# Pilih default vs tuned berdasarkan skor CV (BUKAN skor test), supaya test set tetap murni untuk evaluasi akhir.
cv_default = cv_table.loc[best_name, "cv_macro_f1"]
cv_tuned = search.best_score_
final_clf = tuned if cv_tuned > cv_default else untuned
print(f"CV macro-F1 default={cv_default:.3f} vs tuned={cv_tuned:.3f} -> dipakai: {'tuned' if final_clf is tuned else 'default'}")

test_table = pd.DataFrame({
    n: {"accuracy": accuracy_score(y_te, m.predict(X_te)), "macro_f1": f1_score(y_te, m.predict(X_te), average="macro")}
    for n, m in [("Baseline", dummy), (f"{best_name} (default)", untuned), (f"{best_name} (tuned)", tuned)]
}).T
print("\nEvaluasi akhir pada test set (dipakai SEKALI):")
display(test_table.round(3))
print(classification_report(y_te, final_clf.predict(X_te), digits=3))

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_estimator(final_clf, X_te, y_te, cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Confusion matrix (test set)"); plt.tight_layout(); plt.show()

In [ ]:
# Penjelasan: fitur apa yang paling menentukan segmen? (permutation importance pada test set)
pi = permutation_importance(final_clf, X_te, y_te, n_repeats=20, random_state=RANDOM_STATE, scoring="f1_macro")
pi_df = pd.Series(pi.importances_mean, index=CLUSTER_FEATURES).sort_values()
pi_df.plot(kind="barh", figsize=(7, 5), color="teal"); plt.title("Permutation importance (penurunan macro-F1)")
plt.tight_layout(); plt.show()

# Aturan sederhana yang bisa dibaca manusia (pohon dangkal pada satuan asli)
rules_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=RANDOM_STATE).fit(cust[CLUSTER_FEATURES], cust["segment"])
print(f"Akurasi pohon aturan (depth=3): {rules_tree.score(cust[CLUSTER_FEATURES], cust['segment']):.3f}")
print(export_text(rules_tree, feature_names=CLUSTER_FEATURES))

In [ ]:
# B. Apakah demografi saja (usia + pekerjaan) cukup untuk menebak segmen perilaku?
demo_pipe = lambda m: Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), ["age"]), ("cat", OneHotEncoder(handle_unknown="ignore"), ["occupation"])])),
    ("clf", m),
])
X_demo = cust[["age", "occupation"]]
res = {}
for name, m in {"Baseline (mayoritas)": DummyClassifier(strategy="most_frequent"),
                "Random Forest (demografi)": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)}.items():
    r = cross_validate(demo_pipe(m), X_demo, cust["segment"], cv=cv, scoring={"acc": "accuracy", "f1": "f1_macro"})
    res[name] = {"cv_accuracy": r["test_acc"].mean(), "cv_macro_f1": r["test_f1"].mean(), "cv_f1_std": r["test_f1"].std()}
display(pd.DataFrame(res).T.round(3))

**Cara menafsirkan bagian B:** jika macro-F1 demografi ≈ baseline, artinya *usia dan pekerjaan tidak menjelaskan perilaku transaksi*; tim pemasaran tidak bisa menargetkan segmen berdasarkan data demografi saja.
Itu temuan bisnis yang sah dan berguna. Jika jauh di atas baseline, laporkan fitur demografi mana yang berperan.

## 8. Ekspor artefak untuk dashboard

Semua yang dibutuhkan dashboard disimpan di folder `artifacts/`, bukan `.h5` (format `.h5` milik HDF5/Keras; objek scikit-learn disimpan dengan `.joblib`).

In [ ]:
joblib.dump(seg_pipe, OUT_DIR / "segmentation_pipeline.joblib")
joblib.dump(final_clf, OUT_DIR / "segment_classifier.joblib")

cust_out = cust.reset_index()
cust_out.to_csv(OUT_DIR / "customers_with_segment.csv", index=False)

profile_out = profile.copy()
profile_out.insert(0, "segment_name", profile_out.index.map(SEGMENT_NAMES))
profile_out.insert(1, "n_customers", sizes)
profile_out.to_csv(OUT_DIR / "segment_profile.csv")
z_profile.to_csv(OUT_DIR / "segment_profile_zscore.csv")

tx_out = (tx[["AccountID", "TransactionDate", "TransactionAmount", "Channel", "TransactionType", "Location", "LoginAttempts"]]
          .merge(cust[["segment", "segment_name"]], left_on="AccountID", right_index=True))
tx_out.to_csv(OUT_DIR / "transactions_with_segment.csv", index=False)

metadata = {
    "k": int(K),
    "cluster_features": CLUSTER_FEATURES,
    "log_features": LOG_FEATURES,
    "n_customers": int(len(cust)),
    "silhouette": float(real_sil),
    "silhouette_null_mean": float(null_vals.mean()),
    "stability_ari": float(sweep.loc[K, "stability_ARI"]),
    "classifier": best_name,
    "classifier_test_macro_f1": float(f1_score(y_te, final_clf.predict(X_te), average="macro")),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print(sorted(p.name for p in OUT_DIR.iterdir()))

## 9. Langkah berikutnya

1. Tulis **kesimpulan jujur** di bagian 4 dan nama + rekomendasi segmen di bagian 6.
2. Bangun dashboard di atas folder `artifacts/` (lihat README).
3. Tambahkan bagian *Limitasi*: data sintetis/terbatas, tidak ada label fraud, segmentasi bersifat deskriptif, belum divalidasi lewat A/B test.